In [ ]:
import os
import sys
os.chdir("../..")

import pandas as pd
import duckdb
from pathlib import Path
from config import DATA_ROOT, RESEARCH_ROOT

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import numpy as np
from scipy import stats


import subprocess
subprocess.run(['pip', 'install', 'hmmlearn', '--break-system-packages', '-q'])

from hmmlearn import hmm

In [ ]:
RESEARCH_DB_PATH = DATA_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)

In [4]:
# Load train split, all 8 factors + targets
df = con.execute("""
    SELECT trade_date, nifty_close, fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, up_1d,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct, hmm_state
    FROM daily_features
    WHERE split = 'train'
    ORDER BY trade_date
""").df()

df['trade_date'] = pd.to_datetime(df['trade_date'])

In [6]:
FACTOR_COLS = ['vix_close', 'pcr', 'max_pain_dist_pct', 'basis',
               'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct',
               'client_fut_net_pct']

MIN_PERIODS = 20  # don't z-score off <20 obs; early rows will be NaN and dropped

def expanding_zscore(series, min_periods=MIN_PERIODS):
    """
    z-score using only data up to and including t-1 (no lookahead).
    shift(1) ensures the mean/std at row t exclude the current observation.
    """
    shifted = series.shift(1)
    exp_mean = shifted.expanding(min_periods=min_periods).mean()
    exp_std  = shifted.expanding(min_periods=min_periods).std()
    return (series - exp_mean) / exp_std

In [7]:
# cost_of_carry has structural nulls at monthly expiry/rollover dates,
# same pattern as fut_chng_oi_pct. Forward-fill for consistency before z-scoring.
df['cost_of_carry'] = df['cost_of_carry'].ffill()

print("Raw cost_of_carry NaNs after ffill:", df['cost_of_carry'].isna().sum())

# Rebuild z-scores
df_z = df.copy()
for col in FACTOR_COLS:
    df_z[f'{col}_z'] = expanding_zscore(df[col])

print("\nNaN counts after re-z-scoring:")
print(df_z[[f'{c}_z' for c in FACTOR_COLS]].isna().sum())

Raw cost_of_carry NaNs after ffill: 0

NaN counts after re-z-scoring:
vix_close_z             20
pcr_z                   20
max_pain_dist_pct_z     20
basis_z                 20
cost_of_carry_z         20
fut_chng_oi_pct_z       20
fii_fut_net_pct_z       20
client_fut_net_pct_z    20
dtype: int64


In [ ]:
df_z = df.copy()
for col in FACTOR_COLS:
    df_z[f'{col}_z'] = expanding_zscore(df[col])

# fut_chng_oi_pct is forward-filled at rollover dates upstream — confirm no extra NaNs introduced here
print("NaN counts after z-scoring:")
print(df_z[[f'{c}_z' for c in FACTOR_COLS]].isna().sum())
print(f"\nFirst non-NaN row index (should be ~{MIN_PERIODS}):")
print(df_z[[f'{c}_z' for c in FACTOR_COLS]].apply(lambda c: c.first_valid_index()))

df_z[['trade_date'] + FACTOR_COLS + [f'{c}_z' for c in FACTOR_COLS] + ['hmm_state']].head(25)

NaN counts after z-scoring:
vix_close_z             20
pcr_z                   20
max_pain_dist_pct_z     20
basis_z                 20
cost_of_carry_z         20
fut_chng_oi_pct_z       20
fii_fut_net_pct_z       20
client_fut_net_pct_z    20
dtype: int64

First non-NaN row index (should be ~20):
vix_close_z             20
pcr_z                   20
max_pain_dist_pct_z     20
basis_z                 20
cost_of_carry_z         20
fut_chng_oi_pct_z       20
fii_fut_net_pct_z       20
client_fut_net_pct_z    20
dtype: int64


,trade_date,vix_close,pcr,max_pain_dist_pct,basis,cost_of_carry,fut_chng_oi_pct,fii_fut_net_pct,client_fut_net_pct,vix_close_z,pcr_z,max_pain_dist_pct_z,basis_z,cost_of_carry_z,fut_chng_oi_pct_z,fii_fut_net_pct_z,client_fut_net_pct_z,hmm_state
0,2024-06-21,13.1800,1.0445,0.4528,-13.80,-0.035722,-5.079277,13.92,-7.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
1,2024-06-24,14.0575,1.1381,0.5947,5.35,0.027654,-12.633355,18.15,-9.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2024-06-25,14.3125,1.3802,0.8298,9.95,0.076550,-12.671572,22.89,-14.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,2024-06-26,14.0450,1.3118,0.8615,-0.85,-0.012998,-25.870007,33.89,-18.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2024-06-27,14.1525,1.2555,0.5668,-6.45,-0.012998,-28.670089,63.44,-31.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
5,2024-06-28,13.8025,1.1667,0.5284,121.65,0.068492,1.653298,65.01,-33.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6,2024-07-01,13.8300,1.2051,0.8502,64.30,0.040506,3.049228,65.23,-31.72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
7,2024-07-02,13.6400,1.1506,0.5120,79.30,0.052166,0.093898,64.20,-31.48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
8,2024-07-03,13.2050,1.2384,0.7429,63.90,0.043652,-0.846177,67.23,-34.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
9,2024-07-04,12.8550,1.0307,0.3640,57.30,0.040981,0.429375,67.54,-35.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3


In [9]:
# Drop warm-up rows where any z-scored factor is NaN
df_model = df_z.dropna(subset=[f'{c}_z' for c in FACTOR_COLS]).reset_index(drop=True)
print(f"Rows after dropping warm-up: {len(df_model)} (from {len(df_z)})")

# Confirm hmm_state coverage on the remaining rows
print("\nhmm_state distribution on modeling set:")
print(df_model['hmm_state'].value_counts().sort_index())
print("\nhmm_state NaNs:", df_model['hmm_state'].isna().sum())

Rows after dropping warm-up: 229 (from 249)

hmm_state distribution on modeling set:
hmm_state
0    42
1    35
2    33
3    65
4    54
Name: count, dtype: int64

hmm_state NaNs: 0


In [10]:
# Find expiry/rollover dates within train (same dates where cost_of_carry/fut_chng_oi_pct were null pre-ffill)
expiry_dates = df[df['trade_date'].isin(pd.to_datetime([
    '2024-06-27','2024-07-25','2024-08-29','2024-09-26','2024-10-31',
    '2024-11-28','2024-12-26','2025-01-30','2025-02-27','2025-03-27',
    '2025-04-24','2025-05-29'
]))][['trade_date']]

# row index of each expiry within df_model (post warm-up, post dropna)
for d in expiry_dates['trade_date']:
    idx = df_model[df_model['trade_date'] == d].index
    print(d.date(), idx.tolist())

2024-06-27 []
2024-07-25 [3]
2024-08-29 [27]
2024-09-26 [47]
2024-10-31 [71]
2024-11-28 [89]
2024-12-26 [108]
2025-01-30 [133]
2025-02-27 [153]
2025-03-27 [172]
2025-04-24 [188]
2025-05-29 [212]


In [11]:
from sklearn.linear_model import Ridge

FOLD_BOUNDARIES = [71, 89, 108, 133, 153, 172, 188, 212, 229]
MIN_STATE_OBS = 10
ALPHA = 1.0

Z_COLS = [f'{c}_z' for c in FACTOR_COLS]
TARGET = 'fwd_ret_5d'

results = []         # per-fold diagnostics
pooled_static = []   # pooled out-of-fold predictions, static model
pooled_regime = []   # pooled out-of-fold predictions, regime model
pooled_actual = []   # pooled actuals (aligned)
pooled_dates  = []

train_start = 0
for i, fold_end in enumerate(FOLD_BOUNDARIES[:-1]):
    val_start = fold_end
    val_end   = FOLD_BOUNDARIES[i + 1]

    train_df = df_model.iloc[train_start:fold_end]
    val_df   = df_model.iloc[val_start:val_end]

    if len(val_df) == 0:
        continue

    X_train, y_train = train_df[Z_COLS].values, train_df[TARGET].values
    X_val,   y_val    = val_df[Z_COLS].values,   val_df[TARGET].values

    # --- static ridge (baseline) ---
    static_model = Ridge(alpha=ALPHA)
    static_model.fit(X_train, y_train)
    static_pred = static_model.predict(X_val)

    # --- per-state ridge, with occupancy guard ---
    regime_pred = np.zeros(len(val_df))
    fold_occupancy = {}
    states_used_static_fallback = []

    for state in sorted(df_model['hmm_state'].unique()):
        state_train_mask = train_df['hmm_state'].values == state
        n_state_obs = state_train_mask.sum()
        fold_occupancy[state] = n_state_obs

        val_state_mask = val_df['hmm_state'].values == state
        if val_state_mask.sum() == 0:
            continue  # no rows of this state in validation fold

        if n_state_obs < MIN_STATE_OBS:
            # fallback: use static model's prediction for these rows
            regime_pred[val_state_mask] = static_pred[val_state_mask]
            states_used_static_fallback.append(state)
        else:
            state_model = Ridge(alpha=ALPHA)
            state_model.fit(X_train[state_train_mask], y_train[state_train_mask])
            regime_pred[val_state_mask] = state_model.predict(X_val[val_state_mask])

    # collect pooled predictions
    pooled_static.extend(static_pred.tolist())
    pooled_regime.extend(regime_pred.tolist())
    pooled_actual.extend(y_val.tolist())
    pooled_dates.extend(val_df['trade_date'].tolist())

    results.append({
        'fold': i + 1,
        'train_rows': len(train_df),
        'val_rows': len(val_df),
        'val_start_date': val_df['trade_date'].iloc[0].date(),
        'val_end_date': val_df['trade_date'].iloc[-1].date(),
        'occupancy': fold_occupancy,
        'fallback_states': states_used_static_fallback,
    })

    train_start = 0  # expanding window always starts from 0; only fold_end (train end) grows

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 100)
print(results_df.to_string(index=False))

 fold  train_rows  val_rows val_start_date val_end_date                           occupancy fallback_states
    1          71        18     2024-10-31   2024-11-27  {0: 12, 1: 13, 2: 4, 3: 11, 4: 31}              []
    2          89        19     2024-11-28   2024-12-24  {0: 17, 1: 16, 2: 4, 3: 17, 4: 35}              []
    3         108        25     2024-12-26   2025-01-29  {0: 19, 1: 19, 2: 4, 3: 30, 4: 36}             [2]
    4         133        20     2025-01-30   2025-02-25 {0: 29, 1: 23, 2: 10, 3: 35, 4: 36}              []
    5         153        19     2025-02-27   2025-03-26 {0: 30, 1: 26, 2: 10, 3: 47, 4: 40}              []
    6         172        16     2025-03-27   2025-04-23 {0: 33, 1: 28, 2: 10, 3: 57, 4: 44}              []
    7         188        24     2025-04-24   2025-05-28 {0: 35, 1: 31, 2: 14, 3: 60, 4: 48}              []
    8         212        17     2025-05-29   2025-06-20 {0: 37, 1: 34, 2: 33, 3: 60, 4: 48}              []


In [12]:
from scipy.stats import spearmanr

pooled_df = pd.DataFrame({
    'trade_date': pooled_dates,
    'actual': pooled_actual,
    'pred_static': pooled_static,
    'pred_regime': pooled_regime,
})

def compute_metrics(actual, pred, label):
    ic, p_ic = spearmanr(pred, actual)

    # simple long/short: go long if pred>0, short if pred<0; daily pnl = sign(pred)*actual
    signal = np.sign(pred)
    pnl = signal * actual
    sharpe = pnl.mean() / pnl.std() * np.sqrt(252 / 5)  # annualize 5D-return-based pnl

    hit_rate = (signal * actual > 0).mean()

    # Q5-Q1 spread
    q = pd.qcut(pred, 5, labels=False, duplicates='drop')
    q5 = actual[q == q.max()].mean()
    q1 = actual[q == q.min()].mean()
    spread = q5 - q1

    return {
        'label': label, 'IC': ic, 'IC_p': p_ic,
        'Sharpe': sharpe, 'HitRate': hit_rate, 'Q5_Q1': spread, 'n': len(actual)
    }

m_static = compute_metrics(pooled_df['actual'].values, pooled_df['pred_static'].values, 'Static Ridge')
m_regime = compute_metrics(pooled_df['actual'].values, pooled_df['pred_regime'].values, 'Regime Ridge')

print(pd.DataFrame([m_static, m_regime]).to_string(index=False))

       label       IC     IC_p   Sharpe  HitRate    Q5_Q1   n
Static Ridge 0.178680 0.024688 0.430103 0.525316 1.333081 158
Regime Ridge 0.057757 0.471014 0.093321 0.525316 0.475178 158


In [13]:
print("Static correct signs:", (np.sign(pooled_df['pred_static']) * pooled_df['actual'] > 0).sum())
print("Regime correct signs:", (np.sign(pooled_df['pred_regime']) * pooled_df['actual'] > 0).sum())

# Are the actual sign predictions identical row-by-row, or just same total count?
agree = (np.sign(pooled_df['pred_static']) == np.sign(pooled_df['pred_regime'])).mean()
print(f"Sign agreement between static and regime predictions: {agree:.3f}")

Static correct signs: 83
Regime correct signs: 83
Sign agreement between static and regime predictions: 0.810


In [14]:
fold_ic = []
start = 0
for i, fold_end in enumerate(FOLD_BOUNDARIES[:-1]):
    val_start = fold_end
    val_end = FOLD_BOUNDARIES[i+1]
    n = val_end - val_start
    sl = pooled_df.iloc[start:start+n]
    ic_s, _ = spearmanr(sl['pred_static'], sl['actual'])
    ic_r, _ = spearmanr(sl['pred_regime'], sl['actual'])
    fold_ic.append({'fold': i+1, 'n': n, 'IC_static': ic_s, 'IC_regime': ic_r})
    start += n

print(pd.DataFrame(fold_ic).to_string(index=False))

 fold  n  IC_static  IC_regime
    1 18   0.500516   0.112487
    2 19  -0.152632   0.008772
    3 25   0.395385   0.585385
    4 20  -0.127820   0.087218
    5 19   0.364912  -0.701754
    6 16   0.297059  -0.076471
    7 24   0.247826   0.299130
    8 17   0.132353  -0.095588


In [15]:
n_boot = 2000
rng = np.random.default_rng(42)
n = len(pooled_df)

ic_static_boot, ic_regime_boot = [], []
for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    a = pooled_df['actual'].values[idx]
    s = pooled_df['pred_static'].values[idx]
    r = pooled_df['pred_regime'].values[idx]
    ic_static_boot.append(spearmanr(s, a)[0])
    ic_regime_boot.append(spearmanr(r, a)[0])

ic_static_boot = np.array(ic_static_boot)
ic_regime_boot = np.array(ic_regime_boot)
diff = ic_regime_boot - ic_static_boot

print(f"Static IC 95% CI: [{np.percentile(ic_static_boot,2.5):.3f}, {np.percentile(ic_static_boot,97.5):.3f}]")
print(f"Regime IC 95% CI: [{np.percentile(ic_regime_boot,2.5):.3f}, {np.percentile(ic_regime_boot,97.5):.3f}]")
print(f"Diff (Regime-Static) 95% CI: [{np.percentile(diff,2.5):.3f}, {np.percentile(diff,97.5):.3f}]")
print(f"P(Regime > Static): {(diff>0).mean():.3f}")

Static IC 95% CI: [0.030, 0.326]
Regime IC 95% CI: [-0.105, 0.223]
Diff (Regime-Static) 95% CI: [-0.257, 0.020]
P(Regime > Static): 0.050


In [16]:
SHRINK_K = 20  # fixed shrinkage strength, not tuned per fold
MIN_FIT_OBS = 5  # below this, skip fitting a state model entirely, use global outright

pooled_static2 = []
pooled_shrunk = []
pooled_actual2 = []
pooled_dates2 = []
shrink_diagnostics = []

for i, fold_end in enumerate(FOLD_BOUNDARIES[:-1]):
    val_start = fold_end
    val_end   = FOLD_BOUNDARIES[i + 1]

    train_df = df_model.iloc[0:fold_end]
    val_df   = df_model.iloc[val_start:val_end]
    if len(val_df) == 0:
        continue

    X_train, y_train = train_df[Z_COLS].values, train_df[TARGET].values
    X_val,   y_val    = val_df[Z_COLS].values,   val_df[TARGET].values

    # global (static) model
    global_model = Ridge(alpha=ALPHA)
    global_model.fit(X_train, y_train)
    static_pred = global_model.predict(X_val)
    beta_global = np.concatenate([[global_model.intercept_], global_model.coef_])

    shrunk_pred = np.zeros(len(val_df))
    fold_weights = {}

    for state in sorted(df_model['hmm_state'].unique()):
        state_mask = train_df['hmm_state'].values == state
        n_state = state_mask.sum()
        val_state_mask = val_df['hmm_state'].values == state
        if val_state_mask.sum() == 0:
            continue

        if n_state < MIN_FIT_OBS:
            # not enough to fit at all, use global outright
            shrunk_pred[val_state_mask] = static_pred[val_state_mask]
            fold_weights[state] = 0.0
            continue

        state_model = Ridge(alpha=ALPHA)
        state_model.fit(X_train[state_mask], y_train[state_mask])
        beta_state = np.concatenate([[state_model.intercept_], state_model.coef_])

        w = n_state / (n_state + SHRINK_K)
        beta_shrunk = w * beta_state + (1 - w) * beta_global
        fold_weights[state] = round(w, 3)

        X_val_state = X_val[val_state_mask]
        pred = beta_shrunk[0] + X_val_state @ beta_shrunk[1:]
        shrunk_pred[val_state_mask] = pred

    pooled_static2.extend(static_pred.tolist())
    pooled_shrunk.extend(shrunk_pred.tolist())
    pooled_actual2.extend(y_val.tolist())
    pooled_dates2.extend(val_df['trade_date'].tolist())
    shrink_diagnostics.append({'fold': i+1, 'weights': fold_weights})

print(pd.DataFrame(shrink_diagnostics).to_string(index=False))

pooled_df2 = pd.DataFrame({
    'trade_date': pooled_dates2, 'actual': pooled_actual2,
    'pred_static': pooled_static2, 'pred_shrunk': pooled_shrunk
})

m_static2 = compute_metrics(pooled_df2['actual'].values, pooled_df2['pred_static'].values, 'Static Ridge')
m_shrunk  = compute_metrics(pooled_df2['actual'].values, pooled_df2['pred_shrunk'].values, 'Shrunk Regime Ridge')
print(pd.DataFrame([m_static2, m_shrunk]).to_string(index=False))

 fold                                           weights
    1          {0: 0.375, 1: 0.394, 3: 0.355, 4: 0.608}
    2          {0: 0.459, 1: 0.444, 3: 0.459, 4: 0.636}
    3              {0: 0.487, 1: 0.487, 2: 0.0, 3: 0.6}
    4          {0: 0.592, 1: 0.535, 3: 0.636, 4: 0.643}
    5            {0: 0.6, 1: 0.565, 3: 0.701, 4: 0.667}
    6 {0: 0.623, 1: 0.583, 2: 0.333, 3: 0.74, 4: 0.688}
    7                    {0: 0.636, 1: 0.608, 2: 0.412}
    8            {0: 0.649, 1: 0.63, 3: 0.75, 4: 0.706}
              label       IC     IC_p   Sharpe  HitRate    Q5_Q1   n
       Static Ridge 0.178680 0.024688 0.430103 0.525316 1.333081 158
Shrunk Regime Ridge 0.091099 0.254963 0.491364 0.531646 0.391181 158


In [17]:
n_boot = 2000
rng = np.random.default_rng(42)
n = len(pooled_df2)

ic_static_b, ic_shrunk_b = [], []
for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    a = pooled_df2['actual'].values[idx]
    s = pooled_df2['pred_static'].values[idx]
    r = pooled_df2['pred_shrunk'].values[idx]
    ic_static_b.append(spearmanr(s, a)[0])
    ic_shrunk_b.append(spearmanr(r, a)[0])

diff2 = np.array(ic_shrunk_b) - np.array(ic_static_b)
print(f"Diff (Shrunk-Static) 95% CI: [{np.percentile(diff2,2.5):.3f}, {np.percentile(diff2,97.5):.3f}]")
print(f"P(Shrunk > Static): {(diff2>0).mean():.3f}")

Diff (Shrunk-Static) 95% CI: [-0.196, 0.029]
P(Shrunk > Static): 0.068


In [18]:
FOLD_BOUNDARIES_120 = [133, 153, 172, 188, 212, 229]

def run_walkforward(boundaries, use_shrinkage=False, shrink_k=20, min_fit_obs=5, min_state_obs=10, alpha=1.0):
    pooled = {'date': [], 'actual': [], 'static': [], 'regime': []}
    diag = []
    for i, fold_end in enumerate(boundaries[:-1]):
        val_start, val_end = fold_end, boundaries[i+1]
        train_df = df_model.iloc[0:fold_end]
        val_df = df_model.iloc[val_start:val_end]
        if len(val_df) == 0:
            continue
        X_train, y_train = train_df[Z_COLS].values, train_df[TARGET].values
        X_val, y_val = val_df[Z_COLS].values, val_df[TARGET].values

        global_model = Ridge(alpha=alpha).fit(X_train, y_train)
        static_pred = global_model.predict(X_val)
        beta_global = np.concatenate([[global_model.intercept_], global_model.coef_])

        regime_pred = np.zeros(len(val_df))
        fold_w = {}
        for state in sorted(df_model['hmm_state'].unique()):
            mask = train_df['hmm_state'].values == state
            n_state = mask.sum()
            vmask = val_df['hmm_state'].values == state
            if vmask.sum() == 0:
                continue
            if not use_shrinkage:
                if n_state < min_state_obs:
                    regime_pred[vmask] = static_pred[vmask]
                else:
                    sm = Ridge(alpha=alpha).fit(X_train[mask], y_train[mask])
                    regime_pred[vmask] = sm.predict(X_val[vmask])
            else:
                if n_state < min_fit_obs:
                    regime_pred[vmask] = static_pred[vmask]
                    fold_w[state] = 0.0
                else:
                    sm = Ridge(alpha=alpha).fit(X_train[mask], y_train[mask])
                    beta_state = np.concatenate([[sm.intercept_], sm.coef_])
                    w = n_state / (n_state + shrink_k)
                    beta_shrunk = w * beta_state + (1 - w) * beta_global
                    fold_w[state] = round(w, 3)
                    regime_pred[vmask] = beta_shrunk[0] + X_val[vmask] @ beta_shrunk[1:]

        pooled['date'].extend(val_df['trade_date'].tolist())
        pooled['actual'].extend(y_val.tolist())
        pooled['static'].extend(static_pred.tolist())
        pooled['regime'].extend(regime_pred.tolist())
        diag.append({'fold': i+1, 'train_rows': len(train_df), 'val_rows': len(val_df), 'weights': fold_w})

    return pd.DataFrame(pooled), pd.DataFrame(diag)

pooled_120, diag_120 = run_walkforward(FOLD_BOUNDARIES_120, use_shrinkage=True)
print(diag_120.to_string(index=False))

m_static_120 = compute_metrics(pooled_120['actual'].values, pooled_120['static'].values, 'Static (start=133)')
m_shrunk_120 = compute_metrics(pooled_120['actual'].values, pooled_120['regime'].values, 'Shrunk (start=133)')
print(pd.DataFrame([m_static_120, m_shrunk_120]).to_string(index=False))

 fold  train_rows  val_rows                                           weights
    1         133        20          {0: 0.592, 1: 0.535, 3: 0.636, 4: 0.643}
    2         153        19            {0: 0.6, 1: 0.565, 3: 0.701, 4: 0.667}
    3         172        16 {0: 0.623, 1: 0.583, 2: 0.333, 3: 0.74, 4: 0.688}
    4         188        24                    {0: 0.636, 1: 0.608, 2: 0.412}
    5         212        17            {0: 0.649, 1: 0.63, 3: 0.75, 4: 0.706}
             label        IC     IC_p    Sharpe  HitRate     Q5_Q1  n
Static (start=133)  0.127903 0.214281 -0.643438 0.489583  1.324856 96
Shrunk (start=133) -0.036883 0.721269 -0.402465 0.510417 -0.071325 96


In [19]:
# === STEP 8: TEST SET — RUN ONCE, NO ITERATION AFTER THIS ===

# 1. Load test split (raw, undecoded)
df_test_raw = con.execute("""
    SELECT trade_date, nifty_close, fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, up_1d,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct
    FROM daily_features
    WHERE split = 'test'
    ORDER BY trade_date
""").df()
df_test_raw['trade_date'] = pd.to_datetime(df_test_raw['trade_date'])
df_test_raw['cost_of_carry'] = df_test_raw['cost_of_carry'].ffill()  # same rollover fix as train

print(f"Test rows: {len(df_test_raw)}")
print(f"Test date range: {df_test_raw['trade_date'].min().date()} to {df_test_raw['trade_date'].max().date()}")

# 2. Concatenate train (raw, pre-z-score) + test for continuous expanding z-score
df_full = pd.concat([df[['trade_date'] + FACTOR_COLS]
                      .assign(cost_of_carry=df['cost_of_carry']),  # already ffilled
                      df_test_raw[['trade_date'] + FACTOR_COLS]],
                     ignore_index=True)

for col in FACTOR_COLS:
    df_full[f'{col}_z'] = expanding_zscore(df_full[col])

# split back out — test z-scores now reflect full history up to t-1, no lookahead
n_train = len(df)
df_test_z = df_full.iloc[n_train:].reset_index(drop=True)
df_test_z = pd.concat([df_test_raw.reset_index(drop=True)[['trade_date','fwd_ret_1d','fwd_ret_5d','fwd_ret_20d']],
                        df_test_z[[f'{c}_z' for c in FACTOR_COLS]]], axis=1)

print("\nNaNs in test z-scores (should be 0, since train already warmed up):")
print(df_test_z[[f'{c}_z' for c in FACTOR_COLS]].isna().sum())

Test rows: 247
Test date range: 2025-06-23 to 2026-06-22

NaNs in test z-scores (should be 0, since train already warmed up):
vix_close_z             0
pcr_z                   0
max_pain_dist_pct_z     0
basis_z                 0
cost_of_carry_z         0
fut_chng_oi_pct_z       0
fii_fut_net_pct_z       0
client_fut_net_pct_z    0
dtype: int64


In [20]:
# Verify fut_chng_oi_pct nulls in raw test (should already be ffilled upstream per schema)
print("Raw fut_chng_oi_pct NaNs in test:", df_test_raw['fut_chng_oi_pct'].isna().sum())
print("Raw cost_of_carry NaNs in test (pre-ffill):",
      con.execute("SELECT COUNT(*) FROM daily_features WHERE split='test' AND cost_of_carry IS NULL").fetchone()[0])

# Rebuild df_test_raw WITHOUT pre-ffilling cost_of_carry — do it after concatenation instead
df_test_raw = con.execute("""
    SELECT trade_date, nifty_close, fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, up_1d,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct
    FROM daily_features
    WHERE split = 'test'
    ORDER BY trade_date
""").df()
df_test_raw['trade_date'] = pd.to_datetime(df_test_raw['trade_date'])

# Concatenate train (use UN-ffilled raw cost_of_carry from original df pull, or re-pull raw train)
df_train_raw_cc = con.execute("""
    SELECT trade_date, cost_of_carry FROM daily_features WHERE split='train' ORDER BY trade_date
""").df()
df_train_raw_cc['trade_date'] = pd.to_datetime(df_train_raw_cc['trade_date'])

df_full = pd.concat([
    df[['trade_date'] + [c for c in FACTOR_COLS if c != 'cost_of_carry']].assign(
        cost_of_carry=df_train_raw_cc['cost_of_carry'].values),
    df_test_raw[['trade_date'] + FACTOR_COLS]
], ignore_index=True)

# Now ffill cost_of_carry on the CONTINUOUS series — test's leading NaNs can pull from train's tail
df_full['cost_of_carry'] = df_full['cost_of_carry'].ffill()

print("\ncost_of_carry NaNs after continuous ffill:", df_full['cost_of_carry'].isna().sum())

for col in FACTOR_COLS:
    df_full[f'{col}_z'] = expanding_zscore(df_full[col])

n_train = len(df)
df_test_z = df_full.iloc[n_train:].reset_index(drop=True)
df_test_z = pd.concat([df_test_raw.reset_index(drop=True)[['trade_date','fwd_ret_1d','fwd_ret_5d','fwd_ret_20d']],
                        df_test_z[[f'{c}_z' for c in FACTOR_COLS]]], axis=1)

print("\nNaNs in test z-scores:")
print(df_test_z[[f'{c}_z' for c in FACTOR_COLS]].isna().sum())

Raw fut_chng_oi_pct NaNs in test: 0
Raw cost_of_carry NaNs in test (pre-ffill): 12

cost_of_carry NaNs after continuous ffill: 0

NaNs in test z-scores:
vix_close_z             0
pcr_z                   0
max_pain_dist_pct_z     0
basis_z                 0
cost_of_carry_z         0
fut_chng_oi_pct_z       0
fii_fut_net_pct_z       0
client_fut_net_pct_z    0
dtype: int64


In [23]:
import pickle

with open(RESEARCH_ROOT / 'models/hmm_canonical.pkl', 'rb') as f:
    hmm_artifacts = pickle.load(f)

hmm_model = hmm_artifacts['hmm_model']
hmm_scaler = hmm_artifacts['scaler']
p5 = hmm_artifacts['basis_p5']
p95 = hmm_artifacts['basis_p95']
features_hmm = hmm_artifacts['features_hmm']

print("HMM loaded. n_components:", hmm_model.n_components, "| covariance_type:", hmm_model.covariance_type)
print("Feature order:", features_hmm)

HMM loaded. n_components: 5 | covariance_type: diag
Feature order: ['log_vix', 'basis_w', 'fut_chng_oi_pct']


In [24]:
# === Decode test HMM states using the LOCKED model — no refitting ===
# hmm_model should already exist in your notebook from step 4 (the canonical 5-state model)
# Features: log_vix, basis_w (winsorized p5/p95 from TRAIN), fut_chng_oi_pct — standardized using TRAIN's scaler

# Reconstruct the exact feature matrix the HMM expects, using train's winsorization bounds and scaler
test_log_vix = np.log(df_test_z.merge(df_test_raw[['trade_date','vix_close']], on='trade_date')['vix_close'])
test_basis_w = df_test_raw['basis'].clip(p5, p95).values        # p5/p95 from TRAIN, already defined earlier
test_oi = df_test_raw['fut_chng_oi_pct'].values

# Use the SAME scaler/standardization fitted on train inside step 4 — confirm variable name matches your notebook
# (commonly named `hmm_scaler` if you used sklearn's StandardScaler there)
test_hmm_features = np.column_stack([test_log_vix, test_basis_w, test_oi])
test_hmm_features_scaled = hmm_scaler.transform(test_hmm_features)   # <-- must be TRAIN-fitted scaler, not refit

test_hmm_states = hmm_model.predict(test_hmm_features_scaled)        # <-- locked model, .predict only, no .fit
df_test_z['hmm_state'] = test_hmm_states

print("Test state distribution:")
print(pd.Series(test_hmm_states).value_counts().sort_index())

Test state distribution:
0    84
1    41
2    47
3    23
4    52
Name: count, dtype: int64


c:\Users\sriva\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [27]:
# === ONE-SHOT TEST EVALUATION — fit once on full train, predict once on test ===

X_train_full = df_model[Z_COLS].values
y_train_full = df_model[TARGET].values
X_test = df_test_z[Z_COLS].values
y_test = df_test_z[TARGET].values

# --- Model 1: Static Ridge ---
global_model = Ridge(alpha=ALPHA).fit(X_train_full, y_train_full)
pred_static_test = global_model.predict(X_test)
beta_global = np.concatenate([[global_model.intercept_], global_model.coef_])

# --- Model 2: Plain per-state Ridge (occupancy guard, no shrinkage) ---
pred_plain_test = np.zeros(len(df_test_z))
for state in sorted(df_model['hmm_state'].unique()):
    train_mask = df_model['hmm_state'].values == state
    n_state = train_mask.sum()
    test_mask = df_test_z['hmm_state'].values == state
    if test_mask.sum() == 0:
        continue
    if n_state < MIN_STATE_OBS:
        pred_plain_test[test_mask] = pred_static_test[test_mask]
    else:
        sm = Ridge(alpha=ALPHA).fit(X_train_full[train_mask], y_train_full[train_mask])
        pred_plain_test[test_mask] = sm.predict(X_test[test_mask])

# --- Model 3: Shrunk per-state Ridge ---
pred_shrunk_test = np.zeros(len(df_test_z))
shrink_weights_test = {}
for state in sorted(df_model['hmm_state'].unique()):
    train_mask = df_model['hmm_state'].values == state
    n_state = train_mask.sum()
    test_mask = df_test_z['hmm_state'].values == state
    if test_mask.sum() == 0:
        continue
    if n_state < MIN_FIT_OBS:
        pred_shrunk_test[test_mask] = pred_static_test[test_mask]
        shrink_weights_test[state] = 0.0
    else:
        sm = Ridge(alpha=ALPHA).fit(X_train_full[train_mask], y_train_full[train_mask])
        beta_state = np.concatenate([[sm.intercept_], sm.coef_])
        w = n_state / (n_state + SHRINK_K)
        beta_shrunk = w * beta_state + (1 - w) * beta_global
        shrink_weights_test[state] = round(w, 3)
        pred_shrunk_test[test_mask] = beta_shrunk[0] + X_test[test_mask] @ beta_shrunk[1:]

print("Shrinkage weights used on full train (by state):", shrink_weights_test)
print(f"\nTrain obs per state: {df_model['hmm_state'].value_counts().sort_index().to_dict()}")

# Check how many test rows have undefined fwd_ret_5d (the tail with no future data yet)
n_nan_target = pd.isna(y_test).sum()
print(f"Test rows with NaN fwd_ret_5d: {n_nan_target} / {len(y_test)}")
print(df_test_z.loc[pd.isna(y_test), 'trade_date'].tolist())  # confirm these are the most recent dates

# Filter all three prediction arrays + actual to valid rows only
valid_mask = ~pd.isna(y_test)
y_test_valid = y_test[valid_mask]
pred_static_valid = pred_static_test[valid_mask]
pred_plain_valid  = pred_plain_test[valid_mask]
pred_shrunk_valid = pred_shrunk_test[valid_mask]

print(f"\nValid test rows for evaluation: {valid_mask.sum()} / {len(y_test)}")

m_static_test = compute_metrics(y_test_valid, pred_static_valid, 'Static Ridge (TEST)')
m_plain_test  = compute_metrics(y_test_valid, pred_plain_valid,  'Plain Regime Ridge (TEST)')
m_shrunk_test = compute_metrics(y_test_valid, pred_shrunk_valid, 'Shrunk Regime Ridge (TEST)')

results_test = pd.DataFrame([m_static_test, m_plain_test, m_shrunk_test])
print("\n" + results_test.to_string(index=False))

Shrinkage weights used on full train (by state): {np.int32(0): np.float64(0.677), np.int32(1): np.float64(0.636), np.int32(2): np.float64(0.623), np.int32(3): np.float64(0.765), np.int32(4): np.float64(0.73)}

Train obs per state: {0: 42, 1: 35, 2: 33, 3: 65, 4: 54}
Test rows with NaN fwd_ret_5d: 5 / 247
[Timestamp('2026-06-16 00:00:00'), Timestamp('2026-06-17 00:00:00'), Timestamp('2026-06-18 00:00:00'), Timestamp('2026-06-19 00:00:00'), Timestamp('2026-06-22 00:00:00')]

Valid test rows for evaluation: 242 / 247

                     label        IC     IC_p    Sharpe  HitRate     Q5_Q1   n
       Static Ridge (TEST) -0.036023 0.577076 -0.043742 0.479339 -0.009059 242
 Plain Regime Ridge (TEST) -0.024865 0.700333  0.630765 0.520661  0.599478 242
Shrunk Regime Ridge (TEST) -0.075374 0.242751  0.292188 0.504132  0.335059 242


In [28]:
n_boot = 2000
rng = np.random.default_rng(42)
n = len(y_test_valid)

ic_s, ic_p, ic_sh = [], [], []
for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    a = y_test_valid[idx]
    ic_s.append(spearmanr(pred_static_valid[idx], a)[0])
    ic_p.append(spearmanr(pred_plain_valid[idx], a)[0])
    ic_sh.append(spearmanr(pred_shrunk_valid[idx], a)[0])

for name, arr in [('Static', ic_s), ('Plain', ic_p), ('Shrunk', ic_sh)]:
    arr = np.array(arr)
    print(f"{name} IC 95% CI: [{np.percentile(arr,2.5):.3f}, {np.percentile(arr,97.5):.3f}]")

Static IC 95% CI: [-0.168, 0.103]
Plain IC 95% CI: [-0.158, 0.108]
Shrunk IC 95% CI: [-0.212, 0.062]


In [22]:
con.close()